# Notebook 3: Klasyczne metody uczenia maszynowego (ML)

## Cel
Implementacja i ocena 5 klasycznych algorytmów ML do klasyfikacji EKG:
1. **Regresja logistyczna** (Logistic Regression)
2. **Las losowy** (Random Forest)
3. **Maszyna wektorów nośnych** (SVM z jądrem RBF)
4. **k-Nearest Neighbors** (KNN, k=5)
5. **Gradient Boosting**
6. **Naiwny Bayes** (model bazowy – punkt odniesienia)

Dane wejściowe: wektory 228 cech statystycznych (z Notebook 2).
Ocena bez strojenia hiperparametrów (Kontrola 2).

**Wymaganie:** Kontrola pośrednia nr 2 – wszystkie algorytmy (bez optymalizacji)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, ConfusionMatrixDisplay,
)

sys.path.insert(0, '..')
from src.data_loader import (
    load_ptbxl_metadata, build_labels, load_raw_data,
    get_train_val_test_split, SUPERCLASSES,
)
from src.preprocessing import preprocess_pipeline, preprocess_batch
from src.feature_extraction import extract_features_batch, get_feature_names
from src.classical_models import get_classical_models, evaluate_all_models, train_evaluate_model

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

DATA_PATH = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
FS = 100

print("Moduły załadowane.")


## 1. Wczytanie i przygotowanie danych

Wczytujemy dane, przetwarzamy je potok przetwarzania i ekstrahujemy cechy.
Używamy podzbioru 500 próbek (100 z każdej klasy) do szybkiej demonstracji.
Dla pełnego eksperymentu można usunąć limit `n_per_class`.


In [ ]:
# Wczytaj metadane
Y = load_ptbxl_metadata(DATA_PATH)
Y = build_labels(Y, DATA_PATH)
train_idx, val_idx, test_idx = get_train_val_test_split(Y)

# ── Podzbiór do demonstracji (100 próbek na klasę) ────────────────────────
# Zmień n_per_class=None, żeby użyć PEŁNEGO zbioru (zajmie kilka minut)
N_PER_CLASS = 100

def sample_per_class(df, idx, n=None):
    sampled = []
    for cls in SUPERCLASSES:
        cls_idx = df[(df.label_single == cls) & df.index.isin(idx)].index
        sampled.extend(cls_idx[:n] if n else cls_idx)
    return df.loc[sampled]

Y_train_s = sample_per_class(Y, train_idx, N_PER_CLASS)
Y_test_s  = sample_per_class(Y, test_idx,  N_PER_CLASS // 5)

print(f"Podzbiór treningowy: {len(Y_train_s):,} próbek")
print(f"Podzbiór testowy:    {len(Y_test_s):,} próbek")


In [ ]:
# Wczytaj surowe sygnały i przetwórz
print("Ładowanie sygnałów...")
X_train_raw = load_raw_data(Y_train_s, sampling_rate=FS, data_path=DATA_PATH)
X_test_raw  = load_raw_data(Y_test_s,  sampling_rate=FS, data_path=DATA_PATH)

print("Przetwarzanie potoku...")
X_train_proc = preprocess_batch(X_train_raw, fs=FS, verbose=False)
X_test_proc  = preprocess_batch(X_test_raw,  fs=FS, verbose=False)

print("Ekstrakcja cech...")
X_train = extract_features_batch(X_train_proc, fs=FS, verbose=False)
X_test  = extract_features_batch(X_test_proc,  fs=FS, verbose=False)

# Napraw NaN/Inf
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_test  = np.nan_to_num(X_test,  nan=0.0, posinf=0.0, neginf=0.0)

# Koduj etykiety jako liczby całkowite
le = LabelEncoder()
le.fit(SUPERCLASSES)
y_train = le.transform(Y_train_s['label_single'].values)
y_test  = le.transform(Y_test_s['label_single'].values)

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")
print(f"Klasy: {le.classes_}")


## 2. Trening i ewaluacja modeli

In [ ]:
# Uruchom wszystkie modele i zbierz wyniki
summary_df, all_results = evaluate_all_models(
    X_train, y_train, X_test, y_test,
    classes=list(le.classes_)
)


## 3. Porównanie wyników

In [ ]:
# Tabela porównawcza
print("\n=== TABELA PORÓWNAWCZA MODELI ===")
display_cols = ['Model', 'Accuracy', 'Balanced Accuracy', 'F1 Macro', 'F1 Weighted', 'Train Time [s]']
print(summary_df[display_cols].to_string(index=False))


In [ ]:
# Wykres słupkowy metryk
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
models = summary_df['Model'].tolist()
x = np.arange(len(models))
width = 0.35

# F1-score
axes[0].bar(x - width/2, summary_df['F1 Macro'],    width, label='F1 Macro',    color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, summary_df['F1 Weighted'], width, label='F1 Weighted', color='coral',     alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=25, ha='right')
axes[0].set_ylabel('F1-score')
axes[0].set_title('Porównanie F1-score modeli', fontweight='bold')
axes[0].legend()
axes[0].set_ylim(0, 1)

# Accuracy vs. Balanced Accuracy
axes[1].bar(x - width/2, summary_df['Accuracy'],          width, label='Accuracy',          color='forestgreen', alpha=0.85)
axes[1].bar(x + width/2, summary_df['Balanced Accuracy'], width, label='Balanced Accuracy', color='orange',      alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=25, ha='right')
axes[1].set_ylabel('Dokładność')
axes[1].set_title('Accuracy vs Balanced Accuracy', fontweight='bold')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../results/classical_ml_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Macierze pomyłek (Confusion Matrices)

In [ ]:
# Macierze pomyłek dla wszystkich modeli
n_models = len(all_results)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for ax, (name, result) in zip(axes.flatten(), all_results.items()):
    cm = result['confusion_matrix']
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(f'{name}\nAcc={result["accuracy"]:.3f} | F1={result["f1_macro"]:.3f}',
                 fontweight='bold', fontsize=10)

# Ukryj ostatni pusty wykres jeśli modeli jest mniej niż 6
for i in range(len(all_results), 6):
    axes.flatten()[i].set_visible(False)

plt.tight_layout()
plt.savefig('../results/classical_ml_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()


## 5. Analiza ważności cech (Random Forest)

Las losowy dostarcza miarę ważności cech (feature importance) – które cechy
statystyczne są najbardziej pomocne w klasyfikacji.


In [ ]:
# Feature importance z Random Forest
rf_model = all_results['Random Forest']['model']
importances = rf_model.feature_importances_
feature_names = get_feature_names()

# Top 20 najważniejszych cech
top_n = 20
top_idx = np.argsort(importances)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(
    [feature_names[i] for i in top_idx[::-1]],
    importances[top_idx[::-1]],
    color='steelblue', alpha=0.85
)
ax.set_xlabel('Ważność cechy (Gini impurity)')
ax.set_title(f'Top {top_n} najważniejszych cech – Random Forest', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../results/classical_ml_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nTop 10 najważniejszych cech:")
for rank, i in enumerate(top_idx[:10], 1):
    print(f"  {rank:2d}. {feature_names[i]:35s} = {importances[i]:.4f}")


In [ ]:
# Zapis najlepszego modelu
import pickle
best_model_name = summary_df.iloc[0]['Model']
best_model = all_results[best_model_name]['model']

os.makedirs('../models', exist_ok=True)
with open('../models/best_classical_model.pkl', 'wb') as f:
    pickle.dump({'model': best_model, 'model_name': best_model_name, 'label_encoder': le}, f)

print(f"Najlepszy model: {best_model_name}")
print(f"  F1 Macro: {summary_df.iloc[0]['F1 Macro']:.4f}")
print(f"  Accuracy: {summary_df.iloc[0]['Accuracy']:.4f}")
print(f"Zapisano do: models/best_classical_model.pkl")
